
# S08-demo-03 — Reproducible Model Selection

Этот ноутбук демонстрирует корректный и воспроизводимый процесс выбора лучшей модели классификации.

Что именно показывается:
- единый и заранее зафиксированный протокол эксперимента;
- baseline через `DummyClassifier`;
- сравнение нескольких семейств моделей;
- небольшой и осмысленный `GridSearchCV` только на обучающей части;
- формальное правило выбора лучшей модели по `CV ROC-AUC`;
- финальная диагностика только на `test`;
- сохранение полного комплекта артефактов эксперимента.


In [ ]:

from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.25
CV_SPLITS = 5
ARTIFACTS_DIR = Path('artifacts') / 's08_demo_03_reproducible_model_selection'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_STATE)
print('Artifacts dir:', ARTIFACTS_DIR.resolve())



## 1. Подготовка данных

Используем синтетический, но достаточно реалистичный датасет бинарной классификации:
- часть признаков информативна;
- есть шумовые признаки;
- классы умеренно несбалансированы;
- после генерации специально добавим пропуски.


In [ ]:

X, y = make_classification(
    n_samples=2400,
    n_features=18,
    n_informative=8,
    n_redundant=4,
    n_repeated=0,
    n_classes=2,
    n_clusters_per_class=2,
    weights=[0.60, 0.40],
    class_sep=1.0,
    flip_y=0.02,
    random_state=RANDOM_STATE,
)

feature_names = [f'feature_{i:02d}' for i in range(X.shape[1])]
X_df = pd.DataFrame(X, columns=feature_names)

def inject_missing_values(df: pd.DataFrame, rate: float = 0.06, seed: int = RANDOM_STATE) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    out = df.copy()
    n_rows, n_cols = out.shape
    n_missing = int(n_rows * n_cols * rate)
    rows = rng.integers(0, n_rows, size=n_missing)
    cols = rng.integers(0, n_cols, size=n_missing)
    for r, c in zip(rows, cols):
        out.iat[r, c] = np.nan
    return out

X_df = inject_missing_values(X_df, rate=0.06)

X_train, X_test, y_train, y_test = train_test_split(
    X_df,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print('Train shape:', X_train.shape)
print('Test shape :', X_test.shape)
print('Train class balance:', pd.Series(y_train).value_counts(normalize=True).sort_index().round(3).to_dict())
print('Test class balance :', pd.Series(y_test).value_counts(normalize=True).sort_index().round(3).to_dict())



## 2. Метрики и служебные функции

Все модели будут сравниваться по единому набору метрик. Формальное правило выбора лучшей модели:

**выбираем модель с максимальным средним `CV ROC-AUC` на обучающей части.**

Тестовая выборка не участвует в выборе модели — она нужна только для финальной оценки.


In [ ]:

def evaluate_classifier(model, X, y_true, split_name: str) -> dict:
    y_pred = model.predict(X)
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_score = model.decision_function(X)
    else:
        y_score = y_pred

    return {
        'split': split_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_score),
    }

cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)



## 3. Базовые и кандидатные модели

Сравниваем четыре варианта:
- `DummyClassifier` как sanity baseline;
- `LogisticRegression` внутри `Pipeline` с импутацией и масштабированием;
- `RandomForestClassifier`;
- `HistGradientBoostingClassifier`.

Для каждой модели задаём компактную сетку гиперпараметров, подходящую для семинарской демонстрации.


In [ ]:

search_space = {
    'dummy_most_frequent': {
        'estimator': DummyClassifier(strategy='most_frequent'),
        'param_grid': {},
    },
    'logreg': {
        'estimator': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(max_iter=2000, solver='liblinear', random_state=RANDOM_STATE)),
        ]),
        'param_grid': {
            'model__C': [0.1, 1.0, 3.0, 10.0],
            'model__class_weight': [None, 'balanced'],
        },
    },
    'random_forest': {
        'estimator': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=250)),
        ]),
        'param_grid': {
            'model__max_depth': [None, 6, 10],
            'model__min_samples_leaf': [1, 3, 8],
            'model__max_features': ['sqrt', 0.7],
        },
    },
    'hist_gb': {
        'estimator': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
        ]),
        'param_grid': {
            'model__learning_rate': [0.05, 0.1],
            'model__max_depth': [None, 4, 8],
            'model__max_leaf_nodes': [15, 31],
        },
    },
}

list(search_space.keys())



## 4. Поиск лучших конфигураций на train-only

Каждая модель обучается только на `X_train, y_train`. Для не-baseline моделей используется `GridSearchCV`.

После этого мы фиксируем:
- лучшие гиперпараметры;
- средний `CV ROC-AUC`;
- качество на train;
- качество на test.


In [ ]:

leaderboard_rows = []
cv_results_export = []
best_estimators = {}

for model_name, config in search_space.items():
    estimator = config['estimator']
    param_grid = config['param_grid']

    if param_grid:
        search = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            scoring='roc_auc',
            cv=cv,
            n_jobs=1,
            refit=True,
            return_train_score=True,
        )
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        best_params = search.best_params_
        best_cv_score = search.best_score_

        cv_table = pd.DataFrame(search.cv_results_).sort_values('rank_test_score').copy()
        cv_table.insert(0, 'model_name', model_name)
        cv_results_export.append(cv_table)
    else:
        best_model = estimator.fit(X_train, y_train)
        best_params = {}
        train_metrics = evaluate_classifier(best_model, X_train, y_train, 'train')
        best_cv_score = train_metrics['roc_auc']

    best_estimators[model_name] = best_model

    train_metrics = evaluate_classifier(best_model, X_train, y_train, 'train')
    test_metrics = evaluate_classifier(best_model, X_test, y_test, 'test')

    row = {
        'model_name': model_name,
        'cv_roc_auc_mean': best_cv_score,
        'train_accuracy': train_metrics['accuracy'],
        'train_f1': train_metrics['f1'],
        'train_roc_auc': train_metrics['roc_auc'],
        'test_accuracy': test_metrics['accuracy'],
        'test_f1': test_metrics['f1'],
        'test_roc_auc': test_metrics['roc_auc'],
        'best_params': json.dumps(best_params, ensure_ascii=False),
    }
    leaderboard_rows.append(row)

leaderboard = pd.DataFrame(leaderboard_rows).sort_values(
    by=['cv_roc_auc_mean', 'test_roc_auc'], ascending=False
).reset_index(drop=True)

leaderboard



## 5. Формальный выбор лучшей модели

Подчеркнём ключевой момент: модель выбирается **не** по тестовой метрике, а по среднему значению `CV ROC-AUC` на обучающей части.


In [ ]:

best_model_name = leaderboard.loc[0, 'model_name']
best_model = best_estimators[best_model_name]

print('Selected model:', best_model_name)
print('Selection criterion: max CV ROC-AUC on train folds only')
print('
Leaderboard:')
print(leaderboard[['model_name', 'cv_roc_auc_mean', 'test_roc_auc', 'test_f1']].to_string(index=False))



## 6. Диагностика выбранной модели на test

После того как модель выбрана формально и без использования `test`, можно один раз посмотреть итоговую диагностику на тестовой выборке.


In [ ]:

y_test_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_test_pred)
cm_df = pd.DataFrame(cm, index=['true_0', 'true_1'], columns=['pred_0', 'pred_1'])
cm_df


In [ ]:

selected_test_metrics = evaluate_classifier(best_model, X_test, y_test, 'test')
pd.Series(selected_test_metrics)



## 7. Сохранение артефактов эксперимента

Сохраняем всё, что нужно для воспроизводимости и последующей проверки:
- лидерборд моделей;
- полные результаты CV;
- отчёт о выборе модели;
- метаданные эксперимента;
- предсказания на test;
- сериализованную лучшую модель.


In [ ]:

leaderboard_path = ARTIFACTS_DIR / 'leaderboard.csv'
leaderboard.to_csv(leaderboard_path, index=False)

if cv_results_export:
    cv_results_df = pd.concat(cv_results_export, ignore_index=True)
else:
    cv_results_df = pd.DataFrame()

cv_results_path = ARTIFACTS_DIR / 'cv_results_full.csv'
cv_results_df.to_csv(cv_results_path, index=False)

selection_report = {
    'selected_model_name': best_model_name,
    'selection_rule': 'maximum mean CV ROC-AUC on training folds only',
    'random_state': RANDOM_STATE,
    'cv_splits': CV_SPLITS,
    'test_size': TEST_SIZE,
    'selected_model_test_metrics': selected_test_metrics,
}
with open(ARTIFACTS_DIR / 'selection_report.json', 'w', encoding='utf-8') as f:
    json.dump(selection_report, f, ensure_ascii=False, indent=2)

experiment_meta = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset_rows': int(X_df.shape[0]),
    'dataset_features': int(X_df.shape[1]),
    'train_rows': int(X_train.shape[0]),
    'test_rows': int(X_test.shape[0]),
    'models_considered': list(search_space.keys()),
    'artifact_dir': str(ARTIFACTS_DIR),
}
with open(ARTIFACTS_DIR / 'experiment_meta.json', 'w', encoding='utf-8') as f:
    json.dump(experiment_meta, f, ensure_ascii=False, indent=2)

if hasattr(best_model, 'predict_proba'):
    test_score = best_model.predict_proba(X_test)[:, 1]
elif hasattr(best_model, 'decision_function'):
    test_score = best_model.decision_function(X_test)
else:
    test_score = best_model.predict(X_test)

test_predictions = pd.DataFrame({
    'y_true': y_test,
    'y_pred': y_test_pred,
    'y_score': test_score,
})
test_predictions.to_csv(ARTIFACTS_DIR / 'test_predictions.csv', index=False)

joblib.dump(best_model, ARTIFACTS_DIR / 'best_model.joblib')

print('Saved files:')
for p in sorted(ARTIFACTS_DIR.iterdir()):
    print('-', p.name)



## 8. Проверка воспроизводимости

Повторно загрузим сохранённую модель и убедимся, что её предсказания совпадают с уже посчитанными.


In [ ]:

reloaded_model = joblib.load(ARTIFACTS_DIR / 'best_model.joblib')
reloaded_pred = reloaded_model.predict(X_test)

print('Predictions identical after reload:', np.array_equal(reloaded_pred, y_test_pred))



## 9. Итоги

Главные выводы этого ноутбука:

1. Тестовая выборка не используется для выбора модели.
2. Формальный критерий выбора должен быть зафиксирован заранее.
3. Для воспроизводимости мало просто назвать модель — нужно сохранять параметры, метрики, результаты CV и сам сериализованный объект модели.
4. Инженерный эксперимент считается завершённым только тогда, когда его можно повторить и проверить.
